In [ ]:
import pandas as pd
import re

def ground_truths_escenario3(log_text):
    log = str(log_text).upper()
    
    if 'TYPE=SYSCALL' in log and ('AUID="ROOT"' in log or 'UID="ROOT"' in log) and any(cmd in log for cmd in ['COMM="APT-GET"', 'COMM="SYSTEMCTL"']):
        return ['INFO', 'BAJO', 'INFORMACIÓN']
    elif 'TYPE=SYSCALL' in log and 'COMM="USERADD"' in log:
        return ['MEDIO']
    elif 'ERROR_PARSE' in log or 'FALLO_LLM' in log:
        return ['ERROR']
    else:
        return ['INFO', 'BAJO', 'INFORMACIÓN']

def evaluar_por_orden(path_completo, path_evaluar, funcion_heuristica):
    """
    Evalúa la precisión comparando fila por fila basándose en el orden estricto de los CSV.
    """
    df_completo = pd.read_csv(path_completo)
    df_evaluar = pd.read_csv(path_evaluar)
    
    # Verificación de integridad
    if len(df_completo) != len(df_evaluar):
        print(f"¡ALERTA!: Archivos desfasados. Completo: {len(df_completo)} | A evaluar: {len(df_evaluar)}")
        return
        
    # Construir la lista de la verdad absoluta
    verdad_en_orden = df_completo['Log'].apply(funcion_heuristica).tolist()
    
    aciertos = 0
    total_validos = 0
    
    # Comparar fila por fila
    for i in range(len(df_evaluar)):
        prediccion_llm = str(df_evaluar.loc[i, 'Riesgo']).upper().strip()
        etiquetas_reales = verdad_en_orden[i]
        
        if 'ERROR' not in etiquetas_reales:
            total_validos += 1
            if prediccion_llm in etiquetas_reales:
                aciertos += 1
                
    precision = (aciertos / total_validos) * 100 if total_validos > 0 else 0
    print(verdad_en_orden)
    print(f"Precisión para {path_evaluar}: {precision:.2f}% ({aciertos}/{total_validos})")
    return precision

# --- EJECUCIÓN ---
# Definir la ruta del archivo que tiene la verdad (RAW Completo)
ruta_verdad = '../results/prompt1/escenario3_resultados_raw_completo_phi3mini.csv'

# Evaluar el RAW Completo contra sí mismo para la nota base
evaluar_por_orden(ruta_verdad, ruta_verdad, ground_truths_escenario3)

# Evaluar los otros formatos
evaluar_por_orden(ruta_verdad, '../results/prompt1/escenario3_resultados_raw_reducido_phi3mini.csv', ground_truths_escenario3)
evaluar_por_orden(ruta_verdad, '../results/prompt1/escenario3_resultados_json_reducido_phi3mini.csv', ground_truths_escenario3)

[['INFO', 'BAJO', 'INFORMACIÓN'], ['INFO', 'BAJO', 'INFORMACIÓN'], ['INFO', 'BAJO', 'INFORMACIÓN'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO']]
Precisión para ../results/escenario3_resultados_raw_completo.csv: 10.00% (1/10)
[['INFO', 'BAJO', 'INFORMACIÓN'], ['INFO', 'BAJO', 'INFORMACIÓN'], ['INFO', 'BAJO', 'INFORMACIÓN'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO']]
Precisión para ../results/escenario3_resultados_raw_reducido.csv: 70.00% (7/10)
[['INFO', 'BAJO', 'INFORMACIÓN'], ['INFO', 'BAJO', 'INFORMACIÓN'], ['INFO', 'BAJO', 'INFORMACIÓN'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO'], ['MEDIO']]
Precisión para ../results/escenario3_resultados_json_reducido.csv: 10.00% (1/10)


10.0